# 02 — Data preparation

## Objective

Prepare the 2024 and 2025 CROME data for machine learning.

The model will use information available in 2024 to predict each cell's 2025 land-use classification. This notebook will define the target, select suitable predictors and check for data leakage.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_directory = project_root / "data" / "raw"
interim_data_directory = project_root / "data" / "interim"

print("Project root:", project_root)
print("Raw data directory:", raw_data_directory)
print("Interim data directory:", interim_data_directory)

Project root: /Users/dafyddjones/projects/lincolnshire-crop-prediction
Raw data directory: /Users/dafyddjones/projects/lincolnshire-crop-prediction/data/raw
Interim data directory: /Users/dafyddjones/projects/lincolnshire-crop-prediction/data/interim


## 1. Retrieve the study-area data

Use the same Lincolnshire area examined in the schema audit so that the modelling dataset can be reproduced directly from the official API.

In [5]:
base_api = "https://environment.data.gov.uk/geoservices/datasets"

dataset_id_2024 = "0903079b-35a2-47de-b805-77a0cc0c57bf"
dataset_id_2025 = "04dc895b-e25d-485d-9b0c-d912a0259da8"

collection_2024 = "Crop_Map_of_England_2024_Lincolnshire"
collection_2025 = "Crop_Map_of_England_2025_Lincolnshire"

url_2024 = (
    f"{base_api}/{dataset_id_2024}/ogc/features/v1/"
    f"collections/{collection_2024}/items"
)

url_2025 = (
    f"{base_api}/{dataset_id_2025}/ogc/features/v1/"
    f"collections/{collection_2025}/items"
)

study_bbox = (-0.35, 52.95, -0.30, 53.00)

request_parameters = {
    "f": "json",
    "bbox": ",".join(map(str, study_bbox)),
    "limit": 10_000
}

In [4]:
def retrieve_crome_data(url, parameters):
    response = requests.get(
        url,
        params=parameters,
        timeout=60
    )
    response.raise_for_status()

    geojson_data = response.json()

    geodataframe = gpd.GeoDataFrame.from_features(
        geojson_data["features"],
        crs="EPSG:4326"
    )

    print(
        f"Retrieved {len(geodataframe):,} of "
        f"{geojson_data['numberMatched']:,} matching cells"
    )

    return geodataframe

In [6]:
crome_2024 = retrieve_crome_data(
    url_2024,
    request_parameters
)

crome_2025 = retrieve_crome_data(
    url_2025,
    request_parameters
)

Retrieved 4,868 of 4,868 matching cells
Retrieved 4,868 of 4,868 matching cells


## 2. Create the modelling table

Each row will represent one geographical cell. The 2024 columns will provide potential predictors, while the 2025 land-use code will be the prediction target.

In [7]:
features_2024 = crome_2024[
    ["cromeid", "lucode", "prob", "geometry"]
].rename(
    columns={
        "lucode": "lucode_2024",
        "prob": "confidence_2024"
    }
)

labels_2025 = crome_2025[
    ["cromeid", "lucode", "prob"]
].rename(
    columns={
        "lucode": "target_lucode_2025",
        "prob": "target_confidence_2025"
    }
)

model_data = features_2024.merge(
    labels_2025,
    on="cromeid",
    how="inner",
    validate="one_to_one"
)

model_data = gpd.GeoDataFrame(
    model_data,
    geometry="geometry",
    crs=crome_2024.crs
)

print("Model data shape:", model_data.shape)
model_data.head()

Model data shape: (4868, 6)


,cromeid,lucode_2024,confidence_2024,geometry,target_lucode_2025,target_confidence_2025
0,RPA512626340510,PG01,0.320,"POLYGON ((-0.32484 52.94986, -0.32544 52.94987...",AC66,0.458
1,RPA513046344216,AC66,0.297,"POLYGON ((-0.31699 52.98339, -0.3173 52.98308,...",AC66,0.659
2,RPA512026344112,AC66,0.577,"POLYGON ((-0.33312 52.98237, -0.33341 52.98268...",AC03,0.458
3,RPA512386340579,FA01,0.182,"POLYGON ((-0.32808 52.95084, -0.32839 52.95054...",AC66,0.773
4,RPA513286341168,AC66,0.462,"POLYGON ((-0.31479 52.95564, -0.31539 52.95565...",LG20,0.633


## 3. Check the modelling table

Confirm that the join has not introduced missing values or duplicate cell identifiers.

In [8]:
quality_summary = pd.Series({
    "Rows": len(model_data),
    "Unique CROME IDs": model_data["cromeid"].nunique(),
    "Duplicate CROME IDs": model_data["cromeid"].duplicated().sum(),
    "Missing 2024 codes": model_data["lucode_2024"].isna().sum(),
    "Missing 2025 targets": model_data["target_lucode_2025"].isna().sum(),
    "Missing 2024 confidence": model_data["confidence_2024"].isna().sum(),
    "Missing 2025 confidence": model_data[
        "target_confidence_2025"
    ].isna().sum(),
    "Missing geometries": model_data["geometry"].isna().sum()
})

quality_summary

Rows                       4868
Unique CROME IDs           4868
Duplicate CROME IDs           0
Missing 2024 codes            0
Missing 2025 targets          0
Missing 2024 confidence       0
Missing 2025 confidence       0
Missing geometries            0
dtype: int64

In [9]:
def load_crome_lookup(file_path):
    lookup_raw = pd.read_excel(
        file_path,
        header=None
    )

    lookup = lookup_raw.iloc[1:, :3].copy()

    lookup.columns = [
        "land_cover",
        "lucode",
        "land_use"
    ]

    lookup["land_cover"] = (
        lookup["land_cover"].ffill()
    )

    lookup = lookup.dropna(
        subset=["lucode"]
    )

    return lookup.reset_index(drop=True)

In [10]:
lookup_2024 = load_crome_lookup(
    raw_data_directory
    / "crome_lucode_lookup_2024.xlsx"
)

lookup_2025 = load_crome_lookup(
    raw_data_directory
    / "crome_lucode_lookup_2025.xlsx"
)

print("2024 lookup rows:", len(lookup_2024))
print("2025 lookup rows:", len(lookup_2025))

2024 lookup rows: 88
2025 lookup rows: 41


## 4. Inspect the prediction target

Review the number and frequency of 2025 land-use categories. This will show whether some outcomes are much more common than others.

In [11]:
target_summary = (
    model_data["target_lucode_2025"]
    .value_counts()
    .rename_axis("lucode")
    .reset_index(name="cells")
)

target_summary["percentage"] = (
    target_summary["cells"]
    / len(model_data)
    * 100
).round(2)

target_summary = target_summary.merge(
    lookup_2025[["lucode", "land_use"]],
    on="lucode",
    how="left",
    validate="one_to_one"
)

target_summary = target_summary[
    ["lucode", "land_use", "cells", "percentage"]
]

print(
    "Number of target classes:",
    len(target_summary)
)

target_summary

Number of target classes: 18


,lucode,land_use,cells,percentage
0,AC66,Winter Wheat,2254,46.30
1,CA02,Cover Crop,540,11.09
2,FA01,Fallow Land,294,6.04
3,AC67,Winter Oilseed,289,5.94
4,NA01,Non-vegetated or sparsely-vegetated Land,266,5.46
5,PG01,Grass,252,5.18
6,AC17,Maize,155,3.18
7,LG20,Winter Field beans,153,3.14
8,AC01,Spring Barley,146,3.00
9,AC03,Beet,143,2.94


In [12]:
majority_baseline_accuracy = (
    target_summary.iloc[0]["percentage"]
)

minimum_class_size = 100

rare_target_classes = target_summary[
    target_summary["cells"] < minimum_class_size
]

print(
    "Majority-class baseline:",
    f"{majority_baseline_accuracy:.2f}%"
)

print(
    "Classes with fewer than "
    f"{minimum_class_size} cells:",
    len(rare_target_classes)
)

rare_target_classes

Majority-class baseline: 46.30%
Classes with fewer than 100 cells: 6


,lucode,land_use,cells,percentage
12,TG01,Temporary grassland,85,1.75
13,AC44,Potato,33,0.68
14,AC65,Winter Oats,11,0.23
15,AC27,Strawberry,6,0.12
16,AC19,Spring Oats,1,0.02
17,LG21,Winter Peas,1,0.02


### Rare target classes

Six land-use classes contain fewer than 100 cells. These classes are combined into an `OTHER` category rather than removed. This retains all observations while providing enough examples for each modelling class.

In [14]:
rare_target_codes = set(
    rare_target_classes["lucode"]
)

model_data["target_2025"] = (
    model_data["target_lucode_2025"]
)

rare_target_mask = model_data[
    "target_lucode_2025"
].isin(rare_target_codes)

model_data.loc[
    rare_target_mask,
    "target_2025"
] = "OTHER"

model_target_summary = (
    model_data["target_2025"]
    .value_counts()
    .rename_axis("target")
    .reset_index(name="cells")
)

model_target_summary["percentage"] = (
    model_target_summary["cells"]
    / len(model_data)
    * 100
).round(2)

print(
    "Original target classes:",
    model_data["target_lucode_2025"].nunique()
)

print(
    "Modelling target classes:",
    model_data["target_2025"].nunique()
)

print(
    "Cells grouped as OTHER:",
    rare_target_mask.sum()
)

model_target_summary

Original target classes: 18
Modelling target classes: 13
Cells grouped as OTHER: 137


,target,cells,percentage
0,AC66,2254,46.30
1,CA02,540,11.09
2,FA01,294,6.04
3,AC67,289,5.94
4,NA01,266,5.46
5,PG01,252,5.18
6,AC17,155,3.18
7,LG20,153,3.14
8,AC01,146,3.00
9,AC03,143,2.94


## 5. Create modelling features

The model will use the cell's 2024 land-use code, 2024 classification confidence and geographical location.

The 2025 confidence is excluded because it would not be available when predicting the 2025 classification.

In [15]:
projected_data = model_data.to_crs(
    "EPSG:27700"
)

cell_centroids = (
    projected_data.geometry.centroid
)

model_data["easting"] = cell_centroids.x
model_data["northing"] = cell_centroids.y

feature_columns = [
    "lucode_2024",
    "confidence_2024",
    "easting",
    "northing"
]

X = model_data[feature_columns].copy()
y = model_data["target_2025"].copy()

print("Feature table shape:", X.shape)
print("Target length:", len(y))

X.head()

Feature table shape: (4868, 4)
Target length: 4868


,lucode_2024,confidence_2024,easting,northing
0,PG01,0.320,512626.778605,340512.123903
1,AC66,0.297,513046.696018,344218.716060
2,AC66,0.577,512026.721345,344114.825388
3,FA01,0.182,512386.762061,340581.398936
4,AC66,0.462,513286.771223,341170.282658


In [16]:
land_use_map_2024 = (
    lookup_2024
    .set_index("lucode")["land_use"]
    .to_dict()
)

land_use_map_2025 = (
    lookup_2025
    .set_index("lucode")["land_use"]
    .to_dict()
)

model_data["land_use_2024"] = (
    model_data["lucode_2024"]
    .map(land_use_map_2024)
)

model_data["target_name_2025"] = (
    model_data["target_2025"]
    .map(land_use_map_2025)
    .fillna("Other rare land uses")
)

print(
    "Unmatched 2024 descriptions:",
    model_data["land_use_2024"].isna().sum()
)

print(
    "Unmatched 2025 descriptions:",
    model_data["target_name_2025"].isna().sum()
)

model_data[
    [
        "lucode_2024",
        "land_use_2024",
        "target_2025",
        "target_name_2025"
    ]
].head()

Unmatched 2024 descriptions: 0
Unmatched 2025 descriptions: 0


,lucode_2024,land_use_2024,target_2025,target_name_2025
0,PG01,Grass,AC66,Winter Wheat
1,AC66,Winter Wheat,AC66,Winter Wheat
2,AC66,Winter Wheat,AC03,Beet
3,FA01,Fallow Land,AC66,Winter Wheat
4,AC66,Winter Wheat,LG20,Winter Field beans


## 6. Save the prepared dataset

Save the cleaned features, grouped target and geometry for use in the modelling notebooks. Information that would reveal the 2025 result is excluded from the predictor columns.

In [17]:
prepared_data = model_data[
    [
        "cromeid",
        "lucode_2024",
        "land_use_2024",
        "confidence_2024",
        "easting",
        "northing",
        "target_2025",
        "target_name_2025",
        "geometry"
    ]
].copy()

print("Prepared data shape:", prepared_data.shape)
print(
    "Total missing values:",
    prepared_data.isna().sum().sum()
)

prepared_data.head()

Prepared data shape: (4868, 9)
Total missing values: 0


,cromeid,lucode_2024,land_use_2024,confidence_2024,easting,northing,target_2025,target_name_2025,geometry
0,RPA512626340510,PG01,Grass,0.320,512626.778605,340512.123903,AC66,Winter Wheat,"POLYGON ((-0.32484 52.94986, -0.32544 52.94987..."
1,RPA513046344216,AC66,Winter Wheat,0.297,513046.696018,344218.716060,AC66,Winter Wheat,"POLYGON ((-0.31699 52.98339, -0.3173 52.98308,..."
2,RPA512026344112,AC66,Winter Wheat,0.577,512026.721345,344114.825388,AC03,Beet,"POLYGON ((-0.33312 52.98237, -0.33341 52.98268..."
3,RPA512386340579,FA01,Fallow Land,0.182,512386.762061,340581.398936,AC66,Winter Wheat,"POLYGON ((-0.32808 52.95084, -0.32839 52.95054..."
4,RPA513286341168,AC66,Winter Wheat,0.462,513286.771223,341170.282658,LG20,Winter Field beans,"POLYGON ((-0.31479 52.95564, -0.31539 52.95565..."


In [20]:
output_path = (
    interim_data_directory
    / "crome_prepared_data.gpkg"
)

prepared_data.to_file(
    output_path,
    layer="prepared_data",
    driver="GPKG",
    index=False
)

print("Saved to:", output_path)
print("File exists:", output_path.exists())

Saved to: /Users/dafyddjones/projects/lincolnshire-crop-prediction/data/interim/crome_prepared_data.gpkg
File exists: True


## Conclusion

The prepared dataset contains 4,868 geographical cells and no missing values.

The modelling target contains 13 classes. Six rare land-use categories were combined into an `OTHER` class so that all observations could be retained.

The selected predictors are:

- 2024 land-use code
- 2024 classification confidence
- Easting
- Northing

The 2025 confidence score was excluded from the prepared dataset because it would not be available when making a prediction.

The prepared geographical dataset was saved to `data/interim/crome_prepared_data.gpkg`.